# Results — TSP + MDS + MaxCut

**Este notebook NO entrena / NO resuelve.**

Lee las métricas ya generadas en:
- `Notebooks/Optimization/TSPOptimization/metrics/*_tsp.json`
- `Notebooks/Optimization/MinimumDominatingSetOptimization/metrics/*_mds.json`
- `Notebooks/Optimization/MaxCutOptimization/metrics/*_maxcut.json`

Para cada solver (brute force, greedy, ILP, QUBO+SA, QAOA, VQE) compara **TSP**, **MDS** y **MaxCut**
y promedia solo métricas comparables entre problemas:
- `success_rate`
- `gap_rel = gap_mean / optimal_cost`
- `time_mean_s`

> MaxCut (HamLib + QAOA estilo [demo PennyLane](https://pennylane.ai/demos/tutorial_qaoa_maxcut)):
> `cost` = valor del corte (maximización); el gap es `óptimo − cut`.
>
> No es un meta-análisis estadístico; es un resumen de defensa / documento.
> Fuerza bruta “gana” en tiempo aquí porque las instancias son pequeñas; no escala.


## Cargar métricas TSP, MDS y MaxCut


In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def resolve_optimization_dir() -> Path:
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        root = base / "Notebooks" / "Optimization"
        if root.is_dir():
            return root
    return Path.cwd().resolve() / "Notebooks" / "Optimization"


OPT_DIR = resolve_optimization_dir()
TSP_METRICS = OPT_DIR / "TSPOptimization" / "metrics"
MDS_METRICS = OPT_DIR / "MinimumDominatingSetOptimization" / "metrics"
MAXCUT_METRICS = OPT_DIR / "MaxCutOptimization" / "metrics"
OUT_DIR = OPT_DIR / "ResultsOptimization"
FIGURES_DIR = OUT_DIR / "figures"
METRICS_DIR = OUT_DIR / "metrics"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED = {
    "brute_force": ("bruteforce_tsp.json", "bruteforce_mds.json", "bruteforce_maxcut.json"),
    "greedy": ("greedy_tsp.json", "greedy_mds.json", "greedy_maxcut.json"),
    "ilp": ("ilp_tsp.json", "ilp_mds.json", "ilp_maxcut.json"),
    "simulated_annealing": ("sa_tsp.json", "sa_mds.json", "sa_maxcut.json"),
    "qaoa": ("qaoa_tsp.json", "qaoa_mds.json", "qaoa_maxcut.json"),
    "vqe": ("vqe_tsp.json", "vqe_mds.json", "vqe_maxcut.json"),
}
ALL_SOLVERS = list(EXPECTED.keys())
PROBLEMS = ["tsp", "mds", "maxcut"]
METRIC_DIRS = {"tsp": TSP_METRICS, "mds": MDS_METRICS, "maxcut": MAXCUT_METRICS}


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text())


def row_from_payload(problem: str, solver: str, payload: dict) -> dict:
    q = payload["quality"]
    opt = float(q.get("optimal_cost") or 0.0)
    gap = float(q.get("gap_mean") if q.get("gap_mean") is not None else np.nan)
    gap_rel = float(gap / opt) if opt and not np.isnan(gap) else np.nan
    return {
        "problem": problem,
        "solver": solver,
        "family": payload.get("family", "classical"),
        "success_rate": float(payload["success_rate"]),
        "cost_mean": q.get("cost_mean"),
        "cost_best": q.get("cost_best"),
        "gap_mean": gap,
        "gap_rel": gap_rel,
        "optimal_cost": opt,
        "time_mean_s": float(payload["time_s"]["mean"]),
        "time_std_s": payload["time_s"].get("std"),
        "energy_evals_mean": payload.get("effort", {}).get("energy_evals_mean"),
    }


raw = {p: {} for p in PROBLEMS}
missing = []
for solver, files in EXPECTED.items():
    for problem, filename in zip(PROBLEMS, files):
        path = METRIC_DIRS[problem] / filename
        if not path.exists():
            missing.append(str(path))
        else:
            raw[problem][solver] = load_json(path)

if missing:
    raise FileNotFoundError(
        "Faltan métricas. Ejecuta antes los notebooks TSP, MDS y MaxCut.\n"
        + "\n".join(f" - {m}" for m in missing)
    )

per_rows = []
for solver in ALL_SOLVERS:
    for problem in PROBLEMS:
        per_rows.append(row_from_payload(problem, solver, raw[problem][solver]))

df_per = pd.DataFrame(per_rows)
print(f"Loaded TSP+MDS+MaxCut for {len(ALL_SOLVERS)} solvers")
print(df_per[["problem", "solver", "success_rate", "gap_rel", "time_mean_s"]].round(4).to_string(index=False))


Loaded TSP+MDS+MaxCut for 6 solvers
problem              solver  success_rate  gap_rel  time_mean_s
    tsp         brute_force         1.000   0.0000       0.0001
    mds         brute_force         1.000   0.0000       0.0002
 maxcut         brute_force         1.000   0.0000       0.0003
    tsp              greedy         1.000   0.0000       0.0001
    mds              greedy         0.000   0.5000       0.0000
 maxcut              greedy         1.000   0.0000       0.0001
    tsp                 ilp         1.000   0.0000       0.0704
    mds                 ilp         1.000   0.0000       0.0499
 maxcut                 ilp         1.000   0.0000       0.6565
    tsp simulated_annealing         0.900   0.0188       0.0167
    mds simulated_annealing         1.000   0.0000       0.0068
 maxcut simulated_annealing         1.000   0.0000       0.0033
    tsp                qaoa         1.000   0.0000       1.1217
    mds                qaoa         0.875   0.1250      12.5972
 max

## Tabla consolidada (media TSP + MDS + MaxCut)


In [2]:
mean_rows = []
for solver in ALL_SOLVERS:
    sub = df_per[df_per["solver"] == solver]
    row = {
        "solver": solver,
        "family": sub["family"].iloc[0],
        "problems": PROBLEMS,
        "success_rate": float(sub["success_rate"].mean()),
        "gap_rel": float(sub["gap_rel"].mean()),
        "time_mean_s": float(sub["time_mean_s"].mean()),
    }
    for problem in PROBLEMS:
        s = sub.loc[sub["problem"] == problem].iloc[0]
        row[f"success_{problem}"] = float(s["success_rate"])
        row[f"gap_rel_{problem}"] = float(s["gap_rel"])
        row[f"time_{problem}"] = float(s["time_mean_s"])
    mean_rows.append(row)

df = pd.DataFrame(mean_rows)
print(df.round(4).to_string(index=False))


             solver           family           problems  success_rate  gap_rel  time_mean_s  success_tsp  gap_rel_tsp  time_tsp  success_mds  gap_rel_mds  time_mds  success_maxcut  gap_rel_maxcut  time_maxcut
        brute_force        classical [tsp, mds, maxcut]        1.0000   0.0000       0.0002          1.0       0.0000    0.0001        1.000        0.000    0.0002             1.0             0.0       0.0003
             greedy        classical [tsp, mds, maxcut]        0.6667   0.1667       0.0001          1.0       0.0000    0.0001        0.000        0.500    0.0000             1.0             0.0       0.0001
                ilp        classical [tsp, mds, maxcut]        1.0000   0.0000       0.2589          1.0       0.0000    0.0704        1.000        0.000    0.0499             1.0             0.0       0.6565
simulated_annealing        classical [tsp, mds, maxcut]        0.9667   0.0062       0.0089          0.9       0.0188    0.0167        1.000        0.000    0.0068 

## Estilo y paleta


In [3]:
CLASSICAL_SOLVERS = ["brute_force", "greedy", "ilp", "simulated_annealing"]
QUANTUM_SOLVERS = ["qaoa", "vqe"]

SOLVER_COLORS = {
    "brute_force": "#0B1F33",
    "greedy": "#0E4A75",
    "ilp": "#4A5560",
    "simulated_annealing": "#219BC3",
    "qaoa": "#7EC8E3",
    "vqe": "#A8D5E5",
}
FAMILY_COLORS = {"classical": "#0E4A75", "quantum_inspired": "#A8D5E5"}
PROBLEM_COLORS = {"tsp": "#0E4A75", "mds": "#219BC3", "maxcut": "#7EC8E3", "mean": "#4A5560"}

UNIR_NAVY = "#0E4A75"
UNIR_GRIS = "#858591"
UNIR_CLARO = "#C0C0C8"

plt.rcParams.update({
    "font.sans-serif": ["Arial", "DejaVu Sans", "Helvetica"],
    "axes.edgecolor": UNIR_CLARO,
    "grid.color": UNIR_CLARO,
    "grid.linestyle": "--",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "figure.dpi": 150,
    "legend.fontsize": 9,
})

DISPLAY_NAME = {
    "brute_force": "Brute force",
    "greedy": "Greedy",
    "ilp": "ILP",
    "simulated_annealing": "QUBO + SA",
    "qaoa": "QAOA",
    "vqe": "VQE",
}
PROBLEM_LABEL = {"tsp": "TSP", "mds": "MDS", "maxcut": "MaxCut", "mean": "Media"}

df["display"] = [DISPLAY_NAME[s] for s in df["solver"]]
df_per["display"] = [DISPLAY_NAME[s] for s in df_per["solver"]]


def savefig(name):
    path = FIGURES_DIR / name
    plt.savefig(path, bbox_inches="tight", facecolor="white")
    print("Saved:", path)
    plt.show()


def annotate_bars(ax, container, fmt="{:.2f}", dy=None, fontsize=8):
    for bar in container:
        h = bar.get_height()
        if np.isnan(h):
            continue
        offset = (0.02 * max(ax.get_ylim()[1], 1e-9)) if dy is None else dy
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h + offset,
            fmt.format(h),
            ha="center",
            va="bottom",
            fontsize=fontsize,
            color=UNIR_GRIS,
        )


summary_path = METRICS_DIR / "optimization_mean_summary.csv"
df.to_csv(summary_path, index=False)
per_path = METRICS_DIR / "optimization_per_problem.csv"
df_per.to_csv(per_path, index=False)
print("Saved:", summary_path)
print("Saved:", per_path)


Saved: /workspace/Notebooks/Optimization/ResultsOptimization/metrics/optimization_mean_summary.csv
Saved: /workspace/Notebooks/Optimization/ResultsOptimization/metrics/optimization_per_problem.csv


### 0. TSP vs MDS vs MaxCut vs media — Success rate (slide clave)


In [4]:
fig, ax = plt.subplots(figsize=(12, 5.2))
x = np.arange(len(ALL_SOLVERS))
w = 0.2
offsets = {"tsp": -1.5 * w, "mds": -0.5 * w, "maxcut": 0.5 * w, "mean": 1.5 * w}
series = {
    "tsp": df["success_tsp"],
    "mds": df["success_mds"],
    "maxcut": df["success_maxcut"],
    "mean": df["success_rate"],
}
for key, vals in series.items():
    bars = ax.bar(x + offsets[key], vals, width=w, color=PROBLEM_COLORS[key],
                  edgecolor="white", label=PROBLEM_LABEL[key], zorder=3)
    annotate_bars(ax, bars, fmt="{:.2f}", dy=0.02, fontsize=6)
ax.set_xticks(x)
ax.set_xticklabels([DISPLAY_NAME[s] for s in ALL_SOLVERS], rotation=20, ha="right")
ax.set_ylim(0, 1.2)
ax.set_ylabel("Success rate")
ax.set_title("Optimización — Success: TSP vs MDS vs MaxCut vs media", color=UNIR_NAVY, pad=8)
ax.legend(frameon=False, ncol=4, loc="lower right")
ax.grid(axis="y", zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
savefig("opt_tsp_vs_mds_vs_mean_success.png")


Saved: /workspace/Notebooks/Optimization/ResultsOptimization/figures/opt_tsp_vs_mds_vs_mean_success.png


/tmp/ipykernel_237222/739175348.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 1. Success rate medio por solver


In [5]:
fig, ax = plt.subplots(figsize=(9.5, 4.8))
x = np.arange(len(ALL_SOLVERS))
colors = [SOLVER_COLORS[s] for s in ALL_SOLVERS]
bars = ax.bar(x, df["success_rate"], color=colors, edgecolor="white", zorder=3)
annotate_bars(ax, bars, fmt="{:.2f}", dy=0.02)
ax.set_xticks(x)
ax.set_xticklabels([DISPLAY_NAME[s] for s in ALL_SOLVERS], rotation=20, ha="right")
ax.set_ylim(0, 1.15)
ax.set_ylabel("Success rate (media TSP+MDS+MaxCut)")
ax.set_title("Optimización — Success rate medio por solver", color=UNIR_NAVY, pad=8)
ax.grid(axis="y", zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
savefig("opt_success_rate_mean.png")


Saved: /workspace/Notebooks/Optimization/ResultsOptimization/figures/opt_success_rate_mean.png


/tmp/ipykernel_237222/739175348.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 2. Gap relativo: TSP vs MDS vs MaxCut vs media

`gap_rel = (cost − optimal) / optimal` — comparable entre problemas.


In [6]:
fig, ax = plt.subplots(figsize=(12, 5.2))
x = np.arange(len(ALL_SOLVERS))
w = 0.2
offsets = {"tsp": -1.5 * w, "mds": -0.5 * w, "maxcut": 0.5 * w, "mean": 1.5 * w}
series = {
    "tsp": df["gap_rel_tsp"],
    "mds": df["gap_rel_mds"],
    "maxcut": df["gap_rel_maxcut"],
    "mean": df["gap_rel"],
}
for key, vals in series.items():
    bars = ax.bar(x + offsets[key], vals, width=w, color=PROBLEM_COLORS[key],
                  edgecolor="white", label=PROBLEM_LABEL[key], zorder=3)
    annotate_bars(ax, bars, fmt="{:.2f}", fontsize=6)
ax.axhline(0, color=UNIR_GRIS, lw=1)
ax.set_xticks(x)
ax.set_xticklabels([DISPLAY_NAME[s] for s in ALL_SOLVERS], rotation=20, ha="right")
ax.set_ylabel("Relative optimality gap")
ax.set_title("Optimización — Gap relativo: TSP vs MDS vs MaxCut vs media", color=UNIR_NAVY, pad=8)
ax.legend(frameon=False, ncol=4)
ax.grid(axis="y", zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
savefig("opt_tsp_vs_mds_vs_mean_gap_rel.png")


Saved: /workspace/Notebooks/Optimization/ResultsOptimization/figures/opt_tsp_vs_mds_vs_mean_gap_rel.png


/tmp/ipykernel_237222/739175348.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3. Clásicos vs quantum-inspired (media)


In [7]:
fig, axs = plt.subplots(1, 2, figsize=(10.5, 4.6))
classic = df[df["family"] == "classical"]
quantum = df[df["family"] == "quantum_inspired"]

vals_s = [classic["success_rate"].mean(), quantum["success_rate"].mean()]
b1 = axs[0].bar(
    ["Classical", "Quantum-inspired"],
    vals_s,
    color=[FAMILY_COLORS["classical"], FAMILY_COLORS["quantum_inspired"]],
    edgecolor="#0E4A75",
    linewidth=0.6,
    zorder=3,
)
annotate_bars(axs[0], b1, fmt="{:.2f}", dy=0.02)
axs[0].set_ylim(0, 1.15)
axs[0].set_ylabel("Success rate (media TSP+MDS+MaxCut)")
axs[0].set_title("Success rate", color=UNIR_NAVY)
axs[0].grid(axis="y", zorder=0)
axs[0].spines["top"].set_visible(False)
axs[0].spines["right"].set_visible(False)

vals_g = [classic["gap_rel"].mean(), quantum["gap_rel"].mean()]
b2 = axs[1].bar(
    ["Classical", "Quantum-inspired"],
    vals_g,
    color=[FAMILY_COLORS["classical"], FAMILY_COLORS["quantum_inspired"]],
    edgecolor="#0E4A75",
    linewidth=0.6,
    zorder=3,
)
annotate_bars(axs[1], b2, fmt="{:.3f}")
axs[1].axhline(0, color=UNIR_GRIS, lw=1)
axs[1].set_ylabel("Mean relative gap")
axs[1].set_title("Relative optimality gap", color=UNIR_NAVY)
axs[1].grid(axis="y", zorder=0)
axs[1].spines["top"].set_visible(False)
axs[1].spines["right"].set_visible(False)

fig.suptitle("Optimización — Clásicos vs quantum-inspired (media TSP+MDS+MaxCut)",
             color=UNIR_NAVY, y=1.02, fontsize=13)
fig.tight_layout()
savefig("opt_classical_vs_quantum_mean.png")


Saved: /workspace/Notebooks/Optimization/ResultsOptimization/figures/opt_classical_vs_quantum_mean.png


/tmp/ipykernel_237222/739175348.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4. Success vs runtime (media; puntos por solver)


In [8]:
fig, ax = plt.subplots(figsize=(8.5, 5))
for _, row in df.iterrows():
    t = max(float(row["time_mean_s"]), 1e-6)
    ax.scatter(
        t,
        row["success_rate"],
        s=140,
        color=SOLVER_COLORS[row["solver"]],
        edgecolor="#111111",
        linewidth=0.6,
        zorder=3,
    )
    ax.text(
        t * 1.15,
        row["success_rate"] + 0.02,
        row["display"],
        fontsize=8,
        color=SOLVER_COLORS[row["solver"]],
    )
ax.set_xscale("log")
ax.set_xlabel("Mean time (s, log) — media TSP+MDS+MaxCut")
ax.set_ylabel("Success rate (media)")
ax.set_ylim(-0.05, 1.15)
ax.set_title("Optimización — Success vs runtime (media)", color=UNIR_NAVY, pad=8)
ax.grid(True, which="both", zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
savefig("opt_success_vs_runtime_mean.png")


Saved: /workspace/Notebooks/Optimization/ResultsOptimization/figures/opt_success_vs_runtime_mean.png


/tmp/ipykernel_237222/739175348.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5. Runtime: TSP vs MDS vs MaxCut vs media (log)


In [9]:
fig, ax = plt.subplots(figsize=(12, 5.2))
x = np.arange(len(ALL_SOLVERS))
w = 0.2
offsets = {"tsp": -1.5 * w, "mds": -0.5 * w, "maxcut": 0.5 * w, "mean": 1.5 * w}
series = {
    "tsp": df["time_tsp"].clip(lower=1e-6),
    "mds": df["time_mds"].clip(lower=1e-6),
    "maxcut": df["time_maxcut"].clip(lower=1e-6),
    "mean": df["time_mean_s"].clip(lower=1e-6),
}
for key, vals in series.items():
    ax.bar(x + offsets[key], vals, width=w, color=PROBLEM_COLORS[key],
           edgecolor="white", label=PROBLEM_LABEL[key], zorder=3)
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels([DISPLAY_NAME[s] for s in ALL_SOLVERS], rotation=20, ha="right")
ax.set_ylabel("Mean time (s, log)")
ax.set_title("Optimización — Runtime: TSP vs MDS vs MaxCut vs media", color=UNIR_NAVY, pad=8)
ax.legend(frameon=False, ncol=4)
ax.grid(axis="y", which="both", zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
savefig("opt_runtime_tsp_vs_mds.png")


Saved: /workspace/Notebooks/Optimization/ResultsOptimization/figures/opt_runtime_tsp_vs_mds.png


/tmp/ipykernel_237222/739175348.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6. Heatmap resumen (métricas medias; más oscuro = mejor)


In [10]:
heat_cols = ["success_rate", "gap_rel", "time_mean_s"]
heat_labels = ["Success (mean)", "Gap rel (mean)", "Time mean"]
mat = df.set_index("solver").loc[ALL_SOLVERS, heat_cols]
norm = mat.copy()
for col in heat_cols:
    cmin, cmax = mat[col].min(), mat[col].max()
    if abs(cmax - cmin) < 1e-12:
        norm[col] = 0.5
    elif col in ("gap_rel", "time_mean_s"):
        norm[col] = 1 - (mat[col] - cmin) / (cmax - cmin)
    else:
        norm[col] = (mat[col] - cmin) / (cmax - cmin)

fig, ax = plt.subplots(figsize=(8.2, 4.8))
im = ax.imshow(norm.to_numpy(dtype=float), cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(heat_labels)))
ax.set_xticklabels(heat_labels)
ax.set_yticks(range(len(ALL_SOLVERS)))
ax.set_yticklabels([DISPLAY_NAME[s] for s in ALL_SOLVERS])
ax.set_title("Optimización — Heatmap media TSP+MDS+MaxCut (más oscuro = mejor)", color=UNIR_NAVY, pad=10)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f"{mat.iloc[i, j]:.3g}", ha="center", va="center", color="black", fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
savefig("opt_heatmap_mean.png")


Saved: /workspace/Notebooks/Optimization/ResultsOptimization/figures/opt_heatmap_mean.png


/tmp/ipykernel_237222/739175348.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7. Takeaways (para documento / defensa)

1. **Tres problemas, mismas familias de solvers.** TSP (ruta), MDS (cobertura) y MaxCut (partición; HamLib + QAOA tipo demo PennyLane).
2. **Comparar con métricas relativas.** Costes absolutos no se promedian; sí `success_rate` y `gap_rel`.
3. **Brute force parece “mejor” en tiempo** porque las POCs son instancias pequeñas; no escala ($2^n$ / factorial).
4. **Quantum-inspired (QAOA/VQE)** iguala o se acerca al óptimo en calidad, con **más runtime** que ILP/SA en estas tallas.
5. El valor del experimento es el **protocolo justo**, no un veredicto de ventaja a esta escala.


## Archivos generados


In [11]:
print("Summary / detail tables:")
for p in sorted(METRICS_DIR.glob("*.csv")):
    print(" -", p)
print("\nFigures:")
for p in sorted(FIGURES_DIR.glob("opt_*.png")):
    print(" -", p)


Summary / detail tables:
 - /workspace/Notebooks/Optimization/ResultsOptimization/metrics/maxcut_optimization_summary.csv
 - /workspace/Notebooks/Optimization/ResultsOptimization/metrics/mds_optimization_summary.csv
 - /workspace/Notebooks/Optimization/ResultsOptimization/metrics/optimization_mean_summary.csv
 - /workspace/Notebooks/Optimization/ResultsOptimization/metrics/optimization_per_problem.csv
 - /workspace/Notebooks/Optimization/ResultsOptimization/metrics/tsp_optimization_summary.csv

Figures:
 - /workspace/Notebooks/Optimization/ResultsOptimization/figures/opt_classical_vs_quantum_mean.png
 - /workspace/Notebooks/Optimization/ResultsOptimization/figures/opt_heatmap_mean.png
 - /workspace/Notebooks/Optimization/ResultsOptimization/figures/opt_runtime_tsp_vs_mds.png
 - /workspace/Notebooks/Optimization/ResultsOptimization/figures/opt_success_rate_mean.png
 - /workspace/Notebooks/Optimization/ResultsOptimization/figures/opt_success_vs_runtime_mean.png
 - /workspace/Notebooks/Op